In [619]:
import pandas as pd
import folium
from folium.plugins import BeautifyIcon
import webbrowser
import os
import numpy as np



In [620]:
nodes_final = pd.read_excel("danish_grid_graph_ready.xlsx", sheet_name="nodes")
edges_final = pd.read_excel("danish_grid_graph_ready.xlsx", sheet_name="edges")

In [621]:
def jitter_coordinates(df, lat_col="lat", lon_col="lon", strength=0.001):
    df = df.copy()

    dup_mask = df.duplicated(subset=[lat_col, lon_col], keep=False)

    # Gaussian noise (natural spread)
    df.loc[dup_mask, "lat_jitter"] = (
        df.loc[dup_mask, lat_col] + np.random.normal(0, strength, size=dup_mask.sum())
    )
    df.loc[dup_mask, "lon_jitter"] = (
        df.loc[dup_mask, lon_col] + np.random.normal(0, strength, size=dup_mask.sum())
    )

    # keep original for others
    df.loc[~dup_mask, "lat_jitter"] = df.loc[~dup_mask, lat_col]
    df.loc[~dup_mask, "lon_jitter"] = df.loc[~dup_mask, lon_col]

    return df

In [622]:
nodes_plot = jitter_coordinates(nodes_final, strength=0.01)
m = folium.Map(location=[56.2, 10.0], zoom_start=7, tiles="CartoDB Positron")

In [623]:
def get_node_style(row):
    name = str(row["name"]).lower()
    voltage = row["voltage_kv"]

    if "hvdc" in name:
        return "blue", "circle"
    if "windoff" in name or "havmølle" in name:
        return "yellow", "circle"
    if "værket" in name:
        return "black", "star"

    if pd.notna(voltage):
        if voltage >= 300:
            return "red", "circle"
        elif voltage >= 200:
            return "green", "circle"

    return "gray", "circle"

In [624]:
for _, row in nodes_plot.iterrows():
    if pd.notna(row["lat_jitter"]) and pd.notna(row["lon_jitter"]):

        color, shape = get_node_style(row)

        popup_text = f"""
        <b>{row['name']}</b><br>
        Voltage: {row['voltage_kv']} kV<br>
        Supply: {row['supply']:.1f} MW<br>
        Demand: {row['demand']:.1f} MW<br>
        Source: {row['source']}
        """

        if shape == "star":
            folium.Marker(
                location=[row["lat_jitter"], row["lon_jitter"]],
                icon=BeautifyIcon(
                    icon_shape="star",
                    border_color="black",
                    text_color="black",
                    background_color="white",
                    icon_size=[20, 20]
                ),
                popup=folium.Popup(popup_text, max_width=300)
            ).add_to(m)
        else:
            folium.CircleMarker(
                location=[row["lat_jitter"], row["lon_jitter"]],
                radius=4,
                color=color,
                fill=True,
                fill_opacity=0.8,
                popup=folium.Popup(popup_text, max_width=300)
            ).add_to(m)


In [625]:
def get_edge_style(row):
    voltage = row.get("voltage_kv", None)
    line_type = str(row.get("line_type", "")).lower()
    edge_type = row.get("edge_type", "")

    if edge_type == "hvdc":
        return "blue", "5,5"

    is_cable = "cable" in line_type

    if pd.notna(voltage):
        if voltage >= 300:
            return ("red", "5,5") if is_cable else ("red", None)
        elif voltage >= 200:
            return ("green", "5,5") if is_cable else ("green", None)
        else:
            return ("black", "5,5") if is_cable else ("black", None)

    return "gray", None


for _, row in edges_final.iterrows():
    n1 = nodes_plot[nodes_plot["bus_index"] == row["node1"]]
    n2 = nodes_plot[nodes_plot["bus_index"] == row["node2"]]

    if len(n1) > 0 and len(n2) > 0:
        latlon1 = n1.iloc[0][["lat_jitter", "lon_jitter"]]
        latlon2 = n2.iloc[0][["lat_jitter", "lon_jitter"]]

        if pd.notna(latlon1["lat_jitter"]) and pd.notna(latlon2["lat_jitter"]):

            color, dash = get_edge_style(row)

            folium.PolyLine(
                locations=[
                    [latlon1["lat_jitter"], latlon1["lon_jitter"]],
                    [latlon2["lat_jitter"], latlon2["lon_jitter"]]
                ],
                color=color,
                weight=2,
                opacity=0.7,
                dash_array=dash
            ).add_to(m)

In [626]:
m.save("danish_grid_map.html")
webbrowser.open("file://" + os.path.realpath("danish_grid_map.html"))

True